# Train a Hyperprior Climate-Field Compressor

This notebook trains a Field-Space hyperprior autoencoder in two stages. **Pretraining** teaches the analysis and synthesis transforms to reconstruct multiscale HEALPix fields while learning an initial entropy model. **Joint fine-tuning** starts from the final pretraining checkpoint, freezes the analysis encoder, and optimizes the decoder and entropy model for the final rate–distortion trade-off.

The workflow validates all input stores, computes exact training-split MeanStd statistics, previews the three derived resolutions, runs both Lightning stages with local CSV logging, and plots their losses. Each stage writes a resolved `composed_config.yaml` and `last.ckpt`; the final cell prints a complete entry that can be pasted into the inference notebook's `MODEL_PROFILES` registry. No Weights & Biases account or network logging is used.

## 1. Environment and imports

This cell locates the notebook-local helpers, imports the installed FieldSpaceNN package through the active environment, makes plotting caches writable, fixes random seeds, and reports the selected accelerator.

In [ ]:
import os
import random
import sys
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', '/tmp/fieldspacenn-training-mpl')
os.environ.setdefault('XDG_CACHE_HOME', '/tmp/fieldspacenn-training-xdg')
os.environ["SLURM_JOB_NAME"] = "interactive"
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)
Path(os.environ['XDG_CACHE_HOME']).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from hydra.utils import instantiate
from IPython.display import Markdown, display

search_roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
NOTEBOOK_DIR = next(
    path for root in search_roots for path in (root, root / 'notebooks')
    if (path / 'utils.py').is_file()
)
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
from utils import (
    build_finetuning_config, build_pretraining_config, calculate_normalization,
    inspect_variable_stores, plot_decomposed_input_zooms,
    plot_input_zooms, plot_training_losses,
    prepare_run_directory, run_training_stage, save_config, validate_controls,
    grouped_variable_configuration, split_variable_configuration,
    training_timestep_splits, validation_schedule,
)

# Hydra resolves FieldSpaceNN targets from the package installed in this environment.
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Python: {sys.executable}')
print(f'PyTorch: {torch.__version__}')
print(f'Lightning accelerator selected automatically; currently available: {DEVICE}')

## 2. Editable experiment configuration

This is the only cell most users need to edit. Both stages use `MAX_STEPS` and `BATCH_SIZE`. Batch size is the number of timesteps collated for each optimizer step; increasing it can improve throughput but raises CPU and GPU memory use substantially at high HEALPix resolutions. Supply exactly three distinct HEALPix zooms for `IN_ZOOMS` and define a `2D` group, a `3D` group, or both. A `2D` variable must resolve to one level: use `None` for a time/cell array or one index for a source with a vertical axis. A `3D` variable accepts an arbitrary list of `level_indices`; use `None` to retain every source level. All variables in the `3D` group must resolve to the same number of levels. The optional reserved `timesteps` entry accepts individual non-negative indices and end-exclusive ranges such as `"5-100"`. When provided, it defines the available sample pool; the final 20 selected indices are reserved for validation and testing. `MODEL_COMPLEXITY` scales every attention dimension from 256 and must retain whole 32-channel heads. The `path` must reference to a zarr store that contains HEALPix data at the resolution of the maximum zoom level defined in `IN_ZOOMS`. E.g. for the given example, maximum `IN_ZOOMS` is level 5, the data that `path` references is at HEALPix level 5.

`PROJECT_NAME` and `RUN_NAME` define the location, where checkpoints, preprocessing and configuration files are stored. Existing non-empty run directories are rejected unless `OVERWRITE_EXISTING_RUN` is set to TRUE.

**Optional Data Archives:** Use the <b>Waterpark</b> (<a href="https://waterpark.dkrz.de/da.tabrowser/">https://waterpark.dkrz.de/da.tabrowser/</a>) to access large datasets remotely from our archives at DKRZ. The <b>Waterpark</b> enables fast streaming of zarr archives on the HEALPix grid, which are perfectly suitable for training machine learning models.

**Common failure:** If training runs out of memory, reduce `BATCH_SIZE` before changing the architecture.

In [ ]:
MAX_STEPS = 5000
BATCH_SIZE = 16
IN_ZOOMS = [1, 3, 5]
VARIABLES = {
    # Optional pool: integers and end-exclusive ranges such as '5-100'.
    # 'timesteps': [0, 1, 2, '5-100', 1000],
    '2D': {
        'tas': {
            # OPTIONAL: 'path': 'https://s3.waterpark.dkrz.de/nextgems/healpix/ngc4008/P1D/level_5.zarr'
            'path': '/p/project1/training2640/meuer1/data/nextGEMS_level5.zarr'
        },
    },
    # Every 3D variable must resolve to the same number of levels.
    # Use None for all source levels or provide an arbitrary list of indices.
    # '3D': {
    #     'ua': {
    #         'path': '/path/to/ua_at_the_same_max_zoom.zarr',
    #         'level_indices': [0, 2, 5],
    #     },
    # },
}
MODEL_COMPLEXITY = 1.0

PROJECT_NAME = 'hyperprior_compression_hackathon'
RUN_NAME = 'tas_hpx5'
OVERWRITE_EXISTING_RUN = True
NUM_WORKERS = 8
OUTPUT_ROOT = NOTEBOOK_DIR


## 3. Validate data and define splits

Validation happens before model allocation. The helper checks batch size, paths, 2-D/3-D group membership, equal selected depth within the 3-D group, optional timestep indices, common time/cell geometry, and that the input cell count exactly matches the highest requested zoom. The last 20 samples in the selected pool become two fixed ten-timestep validation/test splits; all earlier selected samples train the model. Without `timesteps`, the complete store is used exactly as before. The displayed interval is also used for validation and periodic checkpoints.

**Common failure:** stores with different timestamps, cell coordinates, or resolutions fail here rather than producing a subtly misaligned model.

In [ ]:
SORTED_ZOOMS, ATTENTION_DIM, VALIDATION_INTERVAL = validate_controls(
    MAX_STEPS, IN_ZOOMS, VARIABLES, MODEL_COMPLEXITY, PROJECT_NAME, RUN_NAME, BATCH_SIZE
)
LOW_ZOOM, MEDIUM_ZOOM, HIGH_ZOOM = SORTED_ZOOMS
inspection = inspect_variable_stores(VARIABLES, expected_zoom=HIGH_ZOOM)
VARIABLE_GROUPS, _ = grouped_variable_configuration(VARIABLES)
VARIABLE_SPECS, _ = split_variable_configuration(VARIABLES)
TIMESTEP_SPLITS = training_timestep_splits(VARIABLES, inspection.n_timesteps)
TRAIN_TIMESTEPS = TIMESTEP_SPLITS['train']
schedule = validation_schedule(MAX_STEPS)
PRETRAIN_DIR = (Path(OUTPUT_ROOT) / 'snapshots' / PROJECT_NAME / RUN_NAME).resolve()
FINETUNE_DIR = (Path(OUTPUT_ROOT) / 'snapshots' / PROJECT_NAME / f'{RUN_NAME}_finetune').resolve()
prepare_run_directory(PRETRAIN_DIR, OVERWRITE_EXISTING_RUN)
prepare_run_directory(FINETUNE_DIR, OVERWRITE_EXISTING_RUN)

display(pd.Series({
    'Variable groups': '; '.join(f'{group}: {", ".join(names)}' for group, names in inspection.variable_groups.items()),
    'Levels per group': str(inspection.group_levels),
    'Input zooms (low/medium/high)': str(SORTED_ZOOMS),
    'Model in_zooms order': str(tuple(reversed(SORTED_ZOOMS))),
    'Highest-resolution cells': f'{inspection.n_cells:,}',
    'Timesteps train / val / test': ' / '.join(str(len(TIMESTEP_SPLITS[name])) for name in ('train', 'val', 'test')),
    'Attention dimension': ATTENTION_DIM,
    'Batch size': BATCH_SIZE,
    'Validation/checkpoint interval': VALIDATION_INTERVAL,
    'Validation steps': str(schedule['validation_steps']),
    'Accelerator policy': 'auto',
}, name='Resolved experiment'))
display(pd.DataFrame(inspection.variables).T)

## 4. Calculate exact normalization statistics

This cell streams every finite value at the selected training indices with float64 combined statistics. Two-dimensional fields receive scalar statistics. Three-dimensional fields receive per-source-level statistics so the loader can safely apply any configured level subset afterward. It always writes a fresh population mean and standard deviation; validation/test values never influence the statistics. If you want to use an existing file for the statistics, store it at the defined `NORM_PATH` and comment out the following lines of code.

**Common failure:** empty/all-missing variables and zero or non-finite standard deviations are rejected because they cannot be normalized safely. On large HPX9 stores this exact pass is intentionally I/O intensive.

In [ ]:
NORM_PATH = PRETRAIN_DIR / 'normalization_mean_std.json'
norm_dict = calculate_normalization(
    VARIABLES, len(TRAIN_TIMESTEPS), NORM_PATH, timesteps=TRAIN_TIMESTEPS
)
display(pd.DataFrame({
    variable: {**entry['stats'], 'finite_count': entry['finite_count']}
    for variable, entry in norm_dict.items()
}).T)
print(f'Normalization written to {NORM_PATH}')

## 5. Build the pretraining configuration

The base HPX5/7/9 architecture is remapped as one coherent unit: `5→low`, `7→medium`, and `9→high`. The highest input path is repeated as the anchor at all three zooms, while `variable_files` preserves the per-variable store mapping. The loader then derives lower resolutions lazily. Group variable counts, group depths, every attention layer's depth-token lengths, total embedding variables, attention/token zooms, down/up blocks, reconstruction losses, samplers, and both grid maxima are updated together.

The resolved YAML saved here is the exact portable input needed by later inference.

**Common failure:** editing the generated config by hand can make checkpoint tensor shapes inconsistent; change the control cell and regenerate instead.

In [ ]:
BASE_CONFIG_PATH = NOTEBOOK_DIR / 'base_config.yaml'
pretraining_cfg = build_pretraining_config(
    BASE_CONFIG_PATH, max_steps=MAX_STEPS, in_zooms=IN_ZOOMS, variables=VARIABLES,
    model_complexity=MODEL_COMPLEXITY, project_name=PROJECT_NAME, run_name=RUN_NAME,
    run_dir=OUTPUT_ROOT, norm_path=NORM_PATH, n_timesteps=inspection.n_timesteps,
    variable_group_levels=inspection.group_levels,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)
PRETRAIN_CONFIG_PATH = save_config(pretraining_cfg, PRETRAIN_DIR / 'composed_config.yaml')
print(f'Pretraining config: {PRETRAIN_CONFIG_PATH}')
print(f'Model input order: {list(pretraining_cfg.model.model.in_zooms)}')
print(f'Passthrough zoom: {list(pretraining_cfg.model.model.passthrough_zooms)}')

## 6. Preview all input resolutions

Now that normalization and the resolved configuration exist, this cell explicitly instantiates the training and validation datasets used by both stages. The first figure composes the normalized sample independently at every requested zoom and is titled **Raw data before decomposition at different resolutions**. The second shows the normalized tensors supplied to the model and is titled **Decomposed data: base field and residuals**. No denormalization is applied by either plotting helper. This is an inspection aid; it does not add an interactive approval gate.

**Common failure:** a shape or merge error usually points to inconsistent per-variable stores or an invalid source level that should have been corrected in the control cell.

In [ ]:
# Instantiate the exact datasets once; both training stages reuse these objects.
training_dataset = instantiate(
    pretraining_cfg.dataloader.dataset, data_dict=pretraining_cfg.data_split.train
)
validation_dataset = instantiate(
    pretraining_cfg.dataloader.dataset, data_dict=pretraining_cfg.data_split.val
)
print(f'Training samples: {len(training_dataset):,}; validation samples: {len(validation_dataset):,}')

# Preview the composed multiresolution fields and the model's decomposed tensors.
COMPOSED_PREVIEW_PATH = PRETRAIN_DIR / 'input_zoom_preview.png'
DECOMPOSED_PREVIEW_PATH = PRETRAIN_DIR / 'input_zoom_decomposition.png'
preview_figure, preview_maps = plot_input_zooms(training_dataset, COMPOSED_PREVIEW_PATH)
plt.show()
decomposition_figure, decomposition_maps = plot_decomposed_input_zooms(
    training_dataset, DECOMPOSED_PREVIEW_PATH
)
plt.show()
for variable, maps in preview_maps.items():
    print(variable, {f'HPX{zoom}': values.shape for zoom, values in maps.items()})
print(f'Composed preview: {COMPOSED_PREVIEW_PATH}')
print(f'Decomposed preview: {DECOMPOSED_PREVIEW_PATH}')

## 7. Stage one: Pretraining

Lightning trains the unfrozen analysis and synthesis transforms with `stage='pretrain'`. Validation and a checkpoint occur approximately every quarter of `MAX_STEPS`; sanity validation is disabled. The training progress bar remains visible, while validation progress bars are suppressed to avoid one output line per validation batch in Jupyter. All scalar logs stay in a local `metrics.csv`, and `last.ckpt` is the handoff artifact.

**Common failure:** HPX9 training can exhaust GPU memory. Reduce `MODEL_COMPLEXITY` in valid 32-channel increments, or first test the workflow with lower zooms; do not alter generated layer dimensions independently.

In [ ]:
pretraining_result = run_training_stage(
    pretraining_cfg, training_dataset, validation_dataset
)
PRETRAIN_CHECKPOINT = pretraining_result['checkpoint']
print(f'Pretraining checkpoint: {PRETRAIN_CHECKPOINT}')
print(f'Local CSV log: {pretraining_result["log_dir"]}')

## 8. Stage two: Joint hyperprior fine-tuning

Fine-tuning is derived by deep-copying the resolved pretraining configuration. It changes the run name to `<RUN_NAME>_finetune`, switches to `stage='joint'`, freezes the analysis encoder, keeps the synthesis decoder trainable, and loads pretrained weights through `ckpt_path_pretrained`. `ckpt_path` remains null, so this is initialization rather than an accidental Lightning resume.

**Common failure:** this stage deliberately refuses to start if pretraining did not produce `last.ckpt`.

In [ ]:
finetuning_cfg = build_finetuning_config(pretraining_cfg, PRETRAIN_CHECKPOINT)
FINETUNE_CONFIG_PATH = save_config(finetuning_cfg, FINETUNE_DIR / 'composed_config.yaml')
finetuning_result = run_training_stage(
    finetuning_cfg, training_dataset, validation_dataset
)
FINETUNE_CHECKPOINT = finetuning_result['checkpoint']
print(f'Fine-tuning config: {FINETUNE_CONFIG_PATH}')
print(f'Fine-tuning checkpoint: {FINETUNE_CHECKPOINT}')
print(f'Local CSV log: {finetuning_result["log_dir"]}')

## 9. Training results and inference handoff

The final cell reads both local Lightning `metrics.csv` files and aligns total training and validation loss against global step using logarithmic loss axes. A third panel shows the estimated training and validation compression ratio during fine-tuning. This estimate comes from the learned entropy probabilities and configured passthrough cost; final artifact compression should still be measured with the inference notebook. The cell also reports final/best validation losses and prints a complete, ready-to-paste `MODEL_PROFILES` entry containing the exact checkpoint, config, zooms, data paths, level selections, and the selected test timesteps when a custom pool was configured.

**Common failure:** if no validation loss or compression ratio appears, confirm that the run reached at least one validation interval and that the CSV contains `val/total_loss` or `*/estimated_compression_ratio`, respectively.

In [ ]:
LOSS_FIGURE_PATH = PRETRAIN_DIR / 'training_loss.png'
loss_figure, loss_summary = plot_training_losses(
    pretraining_result['log_dir'], finetuning_result['log_dir'], LOSS_FIGURE_PATH
)
# Close after explicit display so the inline backend does not render it a second time.
display(loss_figure)
plt.close(loss_figure)
display(pd.DataFrame(loss_summary).T)
PROFILE_KEY = f'{PROJECT_NAME}_{RUN_NAME}_finetune'.lower().replace(' ', '_')
PROFILE_VARIABLES = {
    group: {
        variable: {
            'path': inspection.variables[variable]['path'],
            'level_indices': VARIABLE_GROUPS[group][variable].get('level_indices'),
        }
        for variable in variable_names
    }
    for group, variable_names in inspection.variable_groups.items()
}
if VARIABLES.get('timesteps') is not None:
    PROFILE_VARIABLES = {
        'timesteps': list(pretraining_cfg.data_split.test.timesteps),
        **PROFILE_VARIABLES,
    }
PROFILE_VARIABLE_LINES = ''.join(
    f"            {variable!r}: {specification!r},\n"
    for variable, specification in PROFILE_VARIABLES.items()
)
PROFILE_ENTRY = (
    f"    {PROFILE_KEY!r}: {{\n"
    f"        'label': {f'{RUN_NAME} fine-tuned hyperprior'!r},\n"
    f"        'config_path': Path(r'{FINETUNE_CONFIG_PATH}'),\n"
    f"        'checkpoint_path': Path(r'{FINETUNE_CHECKPOINT}'),\n"
    "        'variables': {\n"
    f"{PROFILE_VARIABLE_LINES}"
    "        },\n"
    f"        'expected_zooms': {list(SORTED_ZOOMS)!r},\n"
    "    },"
)
print('\nCopy this entry inside MODEL_PROFILES in the inference notebook:\n')
print(PROFILE_ENTRY)
print(f'Then set MODEL_PROFILE = {PROFILE_KEY!r}')

## 10. Extension guide

To train another compatible compressor, change only the editable controls and use a new project/run name. Variable and resolution choices are architectural: they affect embeddings, normalization, loss zooms, attention shapes, multiscale mappings, and checkpoint tensors together. That is why inference treats a resolved config and checkpoint as one indivisible model profile.

For a quick resource-conscious smoke run, use `IN_ZOOMS=[1, 3, 5]`, `MODEL_COMPLEXITY=0.125`, and `MAX_STEPS=4`. For scientific training, restore an appropriate complexity and step budget after the complete two-stage pipeline has been verified.